<a href="https://colab.research.google.com/github/ilfpns/PhoSem/blob/main/PhoSem_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhoSem 추론 — 이미지 한 장 → 불량 점수

| 순서 | 셀 | 하는 일 |
|---|---|---|
| 1 | 환경 확인 | 라이브러리 버전과 GPU 여부 출력 |
| 2 | 모델 파일 준비 | `phosem_patchcore_v1.pt`를 찾거나 내려받아 불러오기 |
| 3 | 모델 구성 | 학습 때와 같은 특징 추출기(ResNet50 layer2+3)와 채점 함수 |
| 4 | 이미지 한 장 추론 | `IMAGE_PATH`에 경로를 넣고 실행 → 점수 · 판정 · 히트맵 |
| 5 | 폴더 일괄 추론 (선택) | `INPUT_DIR` 안의 사진 전부 → CSV |

점수가 클수록 정상 die와 멀다는 뜻입니다. 판정 구간은 모델 파일 안의 기준값을 씁니다.

In [ ]:
# @title 1. 환경 확인
import sys
import platform
import importlib.util

REQUIRED = ["torch", "torchvision", "numpy", "pandas", "PIL", "matplotlib"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(f"설치되지 않은 패키지: {missing} → README의 '환경 만들기'를 따라 설치하세요")

import torch
import torchvision
import numpy as np
import pandas as pd
import PIL

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {platform.python_version()} / torch {torch.__version__} / torchvision {torchvision.__version__} / numpy {np.__version__} / pandas {pd.__version__} / Pillow {PIL.__version__}")
print(f"실행 위치: {'Colab' if IN_COLAB else '로컬'} / 장치: {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}")

In [ ]:
# @title 2. 모델 파일 준비
from pathlib import Path

MODEL_NAME = "phosem_patchcore_v1.pt"
MODEL_PATH = None
MODEL_FILE_ID = ""
CANDIDATES = [
    Path("weights") / MODEL_NAME,
    Path(MODEL_NAME),
    Path("/content/drive/MyDrive/300_data/release") / MODEL_NAME,
]

if MODEL_PATH is None and IN_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

if MODEL_PATH is None:
    MODEL_PATH = next((p for p in CANDIDATES if p.exists()), None)
if MODEL_PATH is None and MODEL_FILE_ID:
    import gdown
    Path("weights").mkdir(exist_ok=True)
    MODEL_PATH = Path("weights") / MODEL_NAME
    gdown.download(id=MODEL_FILE_ID, output=str(MODEL_PATH), quiet=False)
if MODEL_PATH is None or not Path(MODEL_PATH).exists():
    raise FileNotFoundError(f"{MODEL_NAME}을 찾지 못했습니다. 드라이브 300_data/release 에서 받아 weights/ 폴더에 넣거나 MODEL_FILE_ID를 채우세요. 찾은 위치: {[str(p) for p in CANDIDATES]}")

pkg = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
if pkg.get("format") != "phosem_patchcore_v1":
    raise ValueError(f"모델 형식이 다릅니다: {pkg.get('format')}")
CFG = pkg["config"]
THR = pkg["thresholds"]
REF = pkg["reference"]
print(f"모델: {MODEL_PATH} / 만든 날 {pkg['created']}")
print(f"메모리 뱅크 {tuple(pkg['memory_bank'].shape)} / 백본 {CFG['backbone']} {CFG['layers']} / 입력 {CFG['img_size']}x{CFG['img_size']} / 특징맵 {CFG['feat_map'][0]}x{CFG['feat_map'][1]} / 점수 {CFG['score']}")
print(f"판정 구간: ≥ {THR['high']:.3f} 불량 의심 / {THR['low']:.3f} ~ {THR['high']:.3f} 검토 필요 / < {THR['low']:.3f} 정상 범위")

In [ ]:
# @title 3. 모델 구성 (학습 때와 같은 특징 추출)
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

IMG_SIZE = int(CFG["img_size"])
FH, FW = [int(v) for v in CFG["feat_map"]]
FEAT_DIM = int(CFG["feat_dim"])
TOP_K = int(CFG["top_k"])
Q_CHUNK = 2048 if DEVICE.type == "cuda" else 512

backbone = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2).eval().to(DEVICE)
TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CFG["normalize_mean"], CFG["normalize_std"]),
])
BANK = pkg["memory_bank"].float().to(DEVICE)
BANK_SQ = (BANK ** 2).sum(1)
NORMAL_REF = REF["normal_scores"].numpy()
TEST_REF = REF["test_scores"].numpy()

def embed(x):
    x = backbone.maxpool(backbone.relu(backbone.bn1(backbone.conv1(x))))
    l2 = backbone.layer2(backbone.layer1(x))
    l3 = backbone.layer3(l2)
    l2 = F.avg_pool2d(l2, 3, 1, 1, count_include_pad=False)
    l3 = F.avg_pool2d(l3, 3, 1, 1, count_include_pad=False)
    l3 = F.interpolate(l3, size=l2.shape[-2:], mode="bilinear", align_corners=False)
    f = torch.cat([l2, l3], 1)
    b, c, h, w = f.shape
    f = f.permute(0, 2, 3, 1).reshape(b * h * w, c)
    return F.adaptive_avg_pool1d(f.unsqueeze(1), FEAT_DIM).squeeze(1), h, w

def judge(score):
    if score >= THR["high"]:
        return "불량 의심"
    if score >= THR["low"]:
        return "검토 필요"
    return "정상 범위"

@torch.no_grad()
def predict(path):
    img = Image.open(path).convert("RGB")
    q, h, w = embed(TF(img).unsqueeze(0).to(DEVICE))
    if (h, w) != (FH, FW):
        raise RuntimeError(f"특징맵 크기 불일치: {(h, w)} != {(FH, FW)}")
    d = torch.empty(q.shape[0], device=DEVICE)
    for s in range(0, q.shape[0], Q_CHUNK):
        qc = q[s:s + Q_CHUNK]
        dd = qc @ BANK.T
        dd.mul_(-2).add_(BANK_SQ.unsqueeze(0)).add_((qc ** 2).sum(1, keepdim=True))
        d[s:s + Q_CHUNK] = dd.min(1).values
    d = d.clamp_min(0).sqrt().cpu()
    score = torch.topk(d, TOP_K).values.mean().item()
    pos = int(d.argmax())
    return {
        "파일": Path(path).name,
        "점수": round(score, 4),
        "판정": judge(score),
        "최댓값": round(float(d.max()), 4),
        "최대위치_y": pos // FW,
        "최대위치_x": pos % FW,
        "정상_백분위": round(float((NORMAL_REF < score).mean() * 100), 1),
        "시험지_순위": f"{int((TEST_REF > score).sum()) + 1} / {len(TEST_REF) + 1}",
        "map": d.reshape(FH, FW).numpy(),
        "image": np.asarray(img),
    }

print(f"준비 완료: 뱅크 {BANK.shape[0]:,}개 조각 / {DEVICE}")

In [ ]:
# @title 4. 이미지 한 장 추론
import matplotlib.pyplot as plt

IMAGE_PATH = ""
OUT_DIR = Path("outputs")

if not IMAGE_PATH and IN_COLAB:
    from google.colab import files
    up = files.upload()
    IMAGE_PATH = next(iter(up))
if not IMAGE_PATH or not Path(IMAGE_PATH).exists():
    raise FileNotFoundError(f"IMAGE_PATH에 사진 경로를 넣으세요: '{IMAGE_PATH}'")

r = predict(IMAGE_PATH)
H, W = r["image"].shape[:2]
m_up = F.interpolate(torch.from_numpy(r["map"])[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()
px, py = (r["최대위치_x"] + 0.5) * W / FW, (r["최대위치_y"] + 0.5) * H / FH

fig, axes = plt.subplots(1, 2, figsize=(8, 5))
axes[0].imshow(r["image"])
axes[0].set_title(r["파일"], fontsize=9)
axes[1].imshow(r["image"])
axes[1].imshow(m_up, cmap="jet", alpha=0.45, vmin=CFG["map_vmin"], vmax=CFG["map_vmax"])
axes[1].scatter([px], [py], marker="x", c="white", s=80, linewidths=2)
axes[1].set_title(f"score {r['점수']:.3f}", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
OUT_DIR.mkdir(exist_ok=True)
fig_path = OUT_DIR / f"{Path(r['파일']).stem}_heatmap.png"
fig.savefig(fig_path, dpi=150)
plt.show()

summary = {k: v for k, v in r.items() if k not in ("map", "image")}
print(pd.Series(summary).to_string())
print(f"\n정상_백분위: 정상 die {len(NORMAL_REF):,}장 중 이 점수보다 낮은 비율 / 시험지_순위: 채점 시험지 {len(TEST_REF):,}장에 넣었을 때 순위 (1위가 가장 불량스러움)")
print(f"저장: {fig_path}")

In [ ]:
# @title 5. 폴더 일괄 추론 (선택)
from tqdm.auto import tqdm

INPUT_DIR = ""
OUT_CSV = Path("outputs") / "predictions.csv"
EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

if not INPUT_DIR:
    print("INPUT_DIR이 비어 있어 건너뜁니다")
else:
    paths = sorted(p for p in Path(INPUT_DIR).iterdir() if p.suffix.lower() in EXTS)
    if not paths:
        raise FileNotFoundError(f"사진이 없습니다: {INPUT_DIR}")
    rows = []
    for p in tqdm(paths, desc="추론"):
        rr = predict(p)
        rows.append({k: v for k, v in rr.items() if k not in ("map", "image")})
    res = pd.DataFrame(rows).sort_values("점수", ascending=False).reset_index(drop=True)
    res.insert(0, "순위", range(1, len(res) + 1))
    OUT_CSV.parent.mkdir(exist_ok=True)
    res.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(res["판정"].value_counts().to_string())
    print(res.head(10).to_string(index=False))
    print(f"\n저장: {OUT_CSV} ({len(res):,}장)")